# Dev/11 Snapshot Integration Tests

This notebook validates the snapshot orchestration layer using repository-local inputs only. It exercises the current backend functionality through snapshot outputs and calls the existing plotting helpers on DataFrames/results produced by repository functions.

In [ ]:
from pathlib import Path
import os
import sys
from types import SimpleNamespace

# Allow execution either from the repository root or from this Dev folder.
if Path.cwd().name != "11_Snapshot":
    candidate = Path.cwd() / "Dev" / "11_Snapshot"
    if candidate.is_dir():
        os.chdir(candidate)

sys.path.insert(0, str(Path.cwd()))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aperture import plot_aperture_envelope, plot_aperture_envelope_with_margin, plot_aperture_source_overlay, plot_margin
from envelope import EnvelopeInputs, plot_envelope, plot_envelope_comparison, plot_sigma
from error_plots import error_table_to_misalignment_offsets, plot_error_table_misalignment_offsets
from orbit_branch import OrbitBranchConfig
from orbit_correction import (
    bpm_measurements_from_twiss,
    enabled_corrector_names,
    normalise_corrector_selection,
    plot_corrector_suggestions,
    plot_orbit_with_bpm,
)
from snapshot import (
    SnapshotConfig,
    SnapshotCorrectorSettings,
    SnapshotOrbitCorrectionConfig,
    SnapshotPlotSaveConfig,
    SnapshotSeriesConfig,
    build_full_cycle_snapshot_series,
    build_machine_snapshot,
    build_snapshot_series,
    config_to_dataframe,
    corrector_settings_from_dataframe,
    corrector_settings_from_manual,
    copy_snapshot_config,
    copy_snapshot_result_config,
    save_snapshot_plot,
    save_snapshot_plots,
    snapshot_configs_from_timepoint_table,
)
from tune_plots import (
    calculate_harmonic_trim_quad_pattern,
    extract_programmed_trim_quad_pattern,
    plot_beta_variation,
    plot_harmonic_trim_quad_pattern,
    plot_resonance_working_points,
    plot_trim_quad_currents,
    plot_tune_diagram_inputs,
)

assert not any("isis" + "_2024" in str(path).lower() for path in Path.cwd().rglob("*.py"))
print(Path.cwd())


## Single Full Snapshot

In [ ]:
base_config = SnapshotConfig(
    cycle_time_ms=0.0,
    requested_qx=4.31,
    requested_qy=3.83,
    label="nominal injection",
    snapshot_id="nominal_injection",
    output_dir="./snapshot_tests/integration",
    run_envelope=True,
    run_aperture=True,
    envelope_inputs=EnvelopeInputs(label="nominal injection", sigma_scale=1.0),
)

snapshot = build_machine_snapshot(base_config)

assert snapshot.machine_state.cycle_time_ms == base_config.cycle_time_ms
assert len(snapshot.table("twiss")) > 100
assert not snapshot.table("madx_summary").empty
assert not snapshot.table("tune_summary").empty
assert {"predicted_qx", "predicted_qy", "dqx", "dqy", "madx_dqx_dpt", "madx_dqy_dpt", "lorentz_beta"}.issubset(snapshot.table("tune_summary").columns)
assert "actual_qx" not in snapshot.table("tune_summary").columns
assert "actual_qy" not in snapshot.table("tune_summary").columns
tune_row = snapshot.table("tune_summary").iloc[0]
assert np.isclose(tune_row["dqx"], tune_row["lorentz_beta"] * tune_row["madx_dqx_dpt"])
assert np.isclose(tune_row["dqy"], tune_row["lorentz_beta"] * tune_row["madx_dqy_dpt"])
assert len(snapshot.table("orbit")) == len(snapshot.table("twiss"))
assert not snapshot.table("orbit_summary").empty
assert not snapshot.table("envelope").empty
assert not snapshot.table("envelope_summary").empty
assert snapshot.envelope_result.inputs.sigma_scale == 1.0
assert not snapshot.table("aperture_aligned").empty
assert not snapshot.table("source_aperture_aligned").empty
assert snapshot.metadata["machine_state"]["requested_qx"] == base_config.requested_qx

assert snapshot.run_paths is not None
snapshot_dir = Path(snapshot.run_paths.snapshot_dir)
assert snapshot_dir.is_dir()
assert snapshot_dir.parent == Path(base_config.output_dir)
assert snapshot_dir.name.endswith(snapshot.snapshot_id)
assert not snapshot.table("corrector_settings").empty
assert not snapshot.table("corrector_summary").empty
assert list(snapshot.table("warnings").columns) == ["timestamp", "snapshot_id", "severity", "source", "code", "message", "context"]
assert "warnings" in snapshot.available_tables()

summary = snapshot.to_summary_dataframe()
assert summary.loc[0, "has_envelope"]
assert summary.loc[0, "has_aperture"]
summary

In [ ]:
manual_settings = corrector_settings_from_manual(
    hd_corrector_currents_A={"r0hd1_kick": 0.05},
    vd_corrector_currents_A={"r0vd1_kick": -0.04},
    prefer="currents",
    source="manual_test",
)
assert isinstance(manual_settings, SnapshotCorrectorSettings)

archiver_corrector_table = pd.DataFrame(
    [
        {"cycle_time_ms": 0.0, "corrector": "r0hd1_kick", "plane": "H", "current_A": 0.05},
        {"cycle_time_ms": 0.0, "corrector": "r0vd1_kick", "plane": "V", "current_A": -0.04},
    ]
)
archiver_settings = corrector_settings_from_dataframe(archiver_corrector_table, cycle_time_ms=0.0)
assert archiver_settings.source == "epics_archiver"

corrector_config = copy_snapshot_config(
    base_config,
    label="archiver corrector injection",
    snapshot_id="archiver_corrector_injection",
    output_dir="./snapshot_tests/correctors",
    run_envelope=False,
    run_aperture=False,
    corrector_settings=archiver_settings,
)
corrector_snapshot = build_machine_snapshot(corrector_config)
corrector_table = corrector_snapshot.table("corrector_settings")
assert {"family", "corrector", "kick_rad", "kick_mrad", "current_A", "source"}.issubset(corrector_table.columns)
assert set(corrector_table["source"]) == {"epics_archiver"}
assert np.isclose(corrector_table.loc[corrector_table["corrector"] == "r0hd1_kick", "current_A"].iloc[0], 0.05)
assert np.isclose(corrector_table.loc[corrector_table["corrector"] == "r0vd1_kick", "current_A"].iloc[0], -0.04)

timepoint_table = pd.DataFrame(
    [
        {"cycle_time_ms": 0.0, "set_qx": 4.31, "set_qy": 3.83, "snapshot_id": "tp_000"},
        {"cycle_time_ms": 0.5, "set_qx": 4.30, "set_qy": 3.82, "snapshot_id": "tp_001", "harmonics": {"D7COS": 0.0005}},
    ]
)
timepoint_correctors = pd.concat(
    [
        archiver_corrector_table,
        archiver_corrector_table.assign(cycle_time_ms=0.5, current_A=lambda df: -df["current_A"]),
    ],
    ignore_index=True,
)
timepoint_configs = snapshot_configs_from_timepoint_table(
    timepoint_table,
    copy_snapshot_config(base_config, output_dir="./snapshot_tests/timepoints", run_envelope=False, run_aperture=False),
    corrector_table=timepoint_correctors,
)
assert len(timepoint_configs) == 2
assert all(config.corrector_settings is not None for config in timepoint_configs)
assert timepoint_configs[1].harmonics == {"D7COS": 0.0005}


## Error Table Misalignment Plots


In [ ]:
survey_error_table = Path("../Error_Tables/jan26_survey_corrected.tfs")
assert survey_error_table.is_file()

misalignment_x = error_table_to_misalignment_offsets(survey_error_table, plane="x")
misalignment_y = error_table_to_misalignment_offsets(survey_error_table, plane="y")
assert len(misalignment_x) == 38
assert len(misalignment_y) == 38
assert {"name", "magnet", "S_start", "S_centre", "S_end", "offset_centre", "angle", "plane"}.issubset(misalignment_x.columns)
assert np.isfinite(misalignment_x[["S_start", "S_centre", "S_end", "offset_centre", "angle"]].to_numpy()).all()
assert np.isfinite(misalignment_y[["S_start", "S_centre", "S_end", "offset_centre", "angle"]].to_numpy()).all()

sp0_qd_x = misalignment_x.loc[misalignment_x["name"] == "SP0_QD"].iloc[0]
sp0_qd_y = misalignment_y.loc[misalignment_y["name"] == "SP0_QD"].iloc[0]
assert np.isclose(sp0_qd_x["offset_centre"], 0.216557548)
assert np.isclose(sp0_qd_y["offset_centre"], 0.286)

sp0_dip = misalignment_x.loc[misalignment_x["name"] == "SP0_DIP"].iloc[0]
assert sp0_dip["S_start"] > 150.0
assert np.isclose(sp0_dip["S_end"], 163.36282)

fig_x, ax_x = plot_error_table_misalignment_offsets(survey_error_table, plane="x", title="Horizontal survey errors")
fig_y, ax_y = plot_error_table_misalignment_offsets(survey_error_table, plane="y", title="Vertical survey errors")
assert ax_x.get_xlabel() == "S [m]"
assert ax_y.get_ylabel() == "Offset [mm]"
plt.close(fig_x)
plt.close(fig_y)


## Existing Plotting Helpers On Snapshot Outputs

In [ ]:
ax = plot_envelope(snapshot.envelope_result, plane="x")
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_envelope(snapshot.envelope_result, plane="y")
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_sigma(snapshot.envelope_result)
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_aperture_envelope(snapshot.aperture_result, plane="x")
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_aperture_envelope_with_margin(snapshot.aperture_result, plane="y")
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_margin(snapshot.aperture_result)
assert ax.figure is not None
plt.show()

In [ ]:
ax = plot_aperture_source_overlay(snapshot.aperture_result, snapshot.source_aperture_result, plane="x")
assert ax.figure is not None
plt.show()

## Copy And Modify Snapshots

In [ ]:
error_config = copy_snapshot_config(
    base_config,
    label="survey error injection",
    snapshot_id="survey_error_injection",
    case="survey_error",
    error_table_paths=["../Error_Tables/jan26_survey_corrected.tfs"],
    run_aperture=False,
)

assert base_config.error_table_paths == []
assert error_config.error_table_paths

roundtrip_config = copy_snapshot_result_config(snapshot, label="retuned copy", snapshot_id="retuned_copy", requested_qx=4.32)
assert roundtrip_config.requested_qx == 4.32
assert snapshot.config.requested_qx == 4.31

config_table = config_to_dataframe([base_config, error_config, roundtrip_config])
assert set(["label", "snapshot_id", "cycle_time_ms"]).issubset(config_table.columns)
config_table[["snapshot_id", "label", "case", "cycle_time_ms", "requested_qx", "requested_qy"]]

## Nominal, Error, And Branch Series

In [ ]:
branch_config = OrbitBranchConfig(
    name="survey_branch",
    lattice_folder="../Lattice_Files/00_Simplified_Lattice",
    error_table_paths=["../Error_Tables/jan26_survey_corrected.tfs"],
)

branch_snapshot_config = copy_snapshot_config(
    base_config,
    label="nominal with branch",
    snapshot_id="nominal_with_branch",
    run_aperture=False,
    branch_configs=[branch_config],
)
branch_snapshot = build_machine_snapshot(branch_snapshot_config)
assert len(branch_snapshot.branch_results) == 1
assert not branch_snapshot.table("branch_comparison").empty

series = build_snapshot_series(
    SnapshotSeriesConfig(
        snapshots=[
            copy_snapshot_config(base_config, run_aperture=False),
            error_config,
        ],
        label="nominal_vs_error",
        output_dir="./snapshot_tests/series",
    )
)

assert len(series.snapshots) == 2
assert series.run_paths is not None
series_dir = Path(series.run_paths.series_dir)
assert series_dir.is_dir()
assert series_dir.parent == Path("./snapshot_tests/series")
assert all(Path(result.run_paths.snapshot_dir).is_relative_to(series_dir) for result in series.snapshots)
assert {"snapshot_id", "label", "case"}.issubset(series.table("summary").columns)
assert {"snapshot_id", "label", "case", "predicted_qx", "predicted_qy"}.issubset(series.table("working_points").columns)
assert not series.table("resonance_lines").empty
assert not series.table("resonance_proximity").empty
assert list(series.table("warnings").columns) == ["timestamp", "snapshot_id", "severity", "source", "code", "message", "context"]
series.table("summary")[["snapshot_id", "case", "cycle_time_ms", "predicted_qx", "predicted_qy"]]

## Full-Cycle Working-Point Resonance Diagram

In [ ]:
cycle_times = [0.0, 2.0, 4.0, 6.0]
set_qx = [4.31, 4.315, 4.32, 4.325]
set_qy = [3.83, 3.835, 3.84, 3.845]
full_cycle_base = copy_snapshot_config(
    base_config,
    run_aperture=False,
    output_dir="./snapshot_tests/full_cycle",
    label="full cycle base",
    snapshot_id="full_cycle_base",
)

full_cycle = build_full_cycle_snapshot_series(
    cycle_times,
    set_qx,
    set_qy,
    base_config=full_cycle_base,
    point_overrides=[
        {"corrector_settings": archiver_settings},
        {"harmonics": {"D7COS": 0.0002}},
        {"error_table_paths": ["../Error_Tables/jan26_survey_corrected.tfs"]},
        {"corrector_settings": manual_settings, "harmonics": {"F8SIN": -0.0003}},
    ],
    label="full_cycle",
    output_dir="./snapshot_tests/full_cycle_series",
)

assert len(full_cycle.snapshots) == len(cycle_times)
assert Path(full_cycle.run_paths.series_dir).parent == Path("./snapshot_tests/full_cycle_series")
assert full_cycle.snapshots[0].config.corrector_settings is not None
assert full_cycle.snapshots[1].config.harmonics == {"D7COS": 0.0002}
assert full_cycle.snapshots[2].config.error_table_paths == ["../Error_Tables/jan26_survey_corrected.tfs"]
assert len(full_cycle.table("working_points")) == len(cycle_times)
assert full_cycle.table("working_points")["predicted_qx"].notna().all()
assert full_cycle.table("working_points")["predicted_qy"].notna().all()
assert full_cycle.table("summary").isna().sum().sum() == 0
assert full_cycle.table("tune_programme").isna().sum().sum() == 0
assert full_cycle.table("working_points").isna().sum().sum() == 0
assert {"iqtf_A", "iqtd_A", "kqtf", "kqtd"}.issubset(full_cycle.table("tune_programme").columns)

full_cycle.table("resonance_proximity")[["snapshot_id", "cycle_time_ms", "nearest_label", "absolute_distance"]]

In [ ]:
fig, ax = plot_tune_diagram_inputs({
    "working_points": full_cycle.table("working_points"),
    "resonance_lines": full_cycle.table("resonance_lines"),
})
assert ax.get_legend() is not None
plt.show()

In [ ]:
fig, ax = plot_resonance_working_points(full_cycle.table("tune_programme"))
assert ax.get_legend() is not None
plt.show()

## Harmonic, Beta, Orbit, And Corrector Plot Helpers

In [ ]:
harmonics = {"D7COS": 0.001, "F8SIN": -0.0015}
harmonic_config = copy_snapshot_config(
    base_config,
    label="harmonic injection",
    snapshot_id="harmonic_injection",
    harmonics=harmonics,
    run_aperture=False,
    output_dir="./snapshot_tests/harmonic",
    envelope_inputs=EnvelopeInputs(label="harmonic injection", sigma_scale=1.0),
)
harmonic_snapshot = build_machine_snapshot(harmonic_config)
assert harmonic_snapshot.envelope_result.inputs.sigma_scale == 1.0

expected_pattern = calculate_harmonic_trim_quad_pattern(harmonics, brho_Tm=harmonic_snapshot.machine_state.beam_state.brho_Tm)
programmed_pattern = extract_programmed_trim_quad_pattern(
    harmonic_snapshot.table("twiss"),
    base_kqtd=harmonic_snapshot.machine_state.kqtd,
    base_kqtf=harmonic_snapshot.machine_state.kqtf,
)
assert not expected_pattern.empty
assert not programmed_pattern.empty

In [ ]:
fig, axes = plot_harmonic_trim_quad_pattern(expected_pattern, programmed_pattern, harmonics=harmonics)
assert len(axes) == 2
plt.show()

In [ ]:
fig, axes = plot_beta_variation(snapshot.table("twiss"), harmonic_snapshot.table("twiss"), harmonics=harmonics)
assert len(axes) == 4
plt.show()


In [ ]:
ax = plot_envelope_comparison([snapshot.envelope_result, harmonic_snapshot.envelope_result], plane="x")
assert ax.figure is not None
plt.show()

In [ ]:
full_cycle.available_tables()

In [ ]:
full_cycle.table('summary')

In [ ]:
full_cycle.table("tune_programme")

In [ ]:
fig, ax = plot_trim_quad_currents(full_cycle.table("tune_programme"))
assert ax.figure is not None
assert len(ax.lines) == 2 or len(ax.patches) == 2
plt.show()

In [ ]:
JAN26_SURVEY_ERROR_TABLE = "../Error_Tables/jan26_survey_corrected.tfs"

distorted_config = copy_snapshot_config(
    base_config,
    label="Jan26 survey closed-orbit distortion",
    snapshot_id="jan26_survey_orbit_distortion",
    case="survey_error",
    run_envelope=False,
    run_aperture=False,
    output_dir="./snapshot_tests/cod",
    error_table_paths=[JAN26_SURVEY_ERROR_TABLE],
)
distorted_snapshot = build_machine_snapshot(distorted_config)
distorted_orbit_summary = distorted_snapshot.table("orbit_summary").iloc[0]
assert distorted_snapshot.metadata["applied_error_tables"] == [JAN26_SURVEY_ERROR_TABLE]
assert distorted_snapshot.machine_state.error_table_path == Path(JAN26_SURVEY_ERROR_TABLE).name
assert distorted_snapshot.metadata["error_table_display"] == Path(JAN26_SURVEY_ERROR_TABLE).name
assert distorted_orbit_summary["max_abs_x_mm"] >= 5.0
assert distorted_orbit_summary["max_abs_y_mm"] >= 5.0

distorted_orbit_summary[["max_abs_x_mm", "max_abs_y_mm", "rms_x_mm", "rms_y_mm"]]


In [ ]:
bpm_measurements_all_h = bpm_measurements_from_twiss(
    distorted_snapshot.table("twiss"),
    plane="H",
    enabled_default=True,
)
assert len(bpm_measurements_all_h) >= 8
assert bpm_measurements_all_h["enabled"].all()
assert bpm_measurements_all_h["closed_orbit_mm"].abs().max() > 0.1

bpm_measurements_h = bpm_measurements_all_h.copy()
bpm_measurements_h["enabled"] = bpm_measurements_h.index % 2 == 0
enabled_bpm_names_h = bpm_measurements_h.loc[bpm_measurements_h["enabled"], "bpm"].tolist()
assert 0 < len(enabled_bpm_names_h) < len(bpm_measurements_all_h)
assert bpm_measurements_h.loc[bpm_measurements_h["enabled"], "s"].max() > 0.8 * bpm_measurements_all_h["s"].max()

ax = plot_orbit_with_bpm(distorted_snapshot.table("twiss"), bpm_measurements_h, plane="H")
assert ax.figure is not None
plt.show()

bpm_measurements_h[["bpm", "s", "closed_orbit_mm", "enabled"]]


In [ ]:
bpm_measurements_all_v = bpm_measurements_from_twiss(
    distorted_snapshot.table("twiss"),
    plane="V",
    enabled_default=True,
)
assert len(bpm_measurements_all_v) >= 8
assert bpm_measurements_all_v["enabled"].all()
assert bpm_measurements_all_v["closed_orbit_mm"].abs().max() > 0.1

bpm_measurements_v = bpm_measurements_all_v.copy()
bpm_measurements_v["enabled"] = bpm_measurements_v.index % 2 == 0
enabled_bpm_names_v = bpm_measurements_v.loc[bpm_measurements_v["enabled"], "bpm"].tolist()
assert 0 < len(enabled_bpm_names_v) < len(bpm_measurements_all_v)
assert bpm_measurements_v.loc[bpm_measurements_v["enabled"], "s"].max() > 0.8 * bpm_measurements_all_v["s"].max()

ax = plot_orbit_with_bpm(distorted_snapshot.table("twiss"), bpm_measurements_v, plane="V")
assert ax.figure is not None
plt.show()

bpm_measurements_v[["bpm", "s", "closed_orbit_mm", "enabled"]]


## Snapshot-Level Read-Only MAD-X Orbit Correction From Jan26 Survey Distortion

This section validates the GUI-facing correction path. The snapshot layer owns the orchestration, but correction physics still runs through `MadxModel` and `orbit_correction.py`. The result exposes suggested currents, corrected TWISS/orbit data, monitor summaries, and one per-BPM before/after comparison table.


In [ ]:
def select_correctors(plane, corrector_names):
    correctors = normalise_corrector_selection(plane=plane)
    selected = {str(name).lower() for name in corrector_names}
    correctors["enabled"] = correctors["corrector"].str.lower().isin(selected)
    assert correctors["enabled"].sum() == len(selected)
    return correctors


h_correctors = select_correctors("H", ["r0hd1_kick", "r3hd1_kick", "r5hd1_kick", "r9hd1_kick"])
v_correctors = select_correctors("V", ["r0vd1_kick", "r3vd1_kick", "r5vd1_kick", "r9vd1_kick"])

correction_config = copy_snapshot_config(
    distorted_config,
    label="Jan26 survey orbit correction suggestions",
    snapshot_id="jan26_survey_orbit_correction",
    output_dir="./snapshot_tests/correction",
    orbit_correction_configs=[
        SnapshotOrbitCorrectionConfig(
            plane="H",
            label="horizontal_selected_bpm",
            bpm_measurements=bpm_measurements_h,
            correctors=h_correctors,
        ),
        SnapshotOrbitCorrectionConfig(
            plane="V",
            label="vertical_selected_bpm",
            bpm_measurements=bpm_measurements_v,
            correctors=v_correctors,
        ),
    ],
)
correction_snapshot = build_machine_snapshot(correction_config)
assert len(correction_snapshot.orbit_correction_results) == 2
assert "orbit_correction_bpm_comparison" in correction_snapshot.available_tables()

summary = correction_snapshot.table("orbit_correction_summary")
correctors = correction_snapshot.table("orbit_correction_correctors")
bpm_compare = correction_snapshot.table("orbit_correction_bpm_comparison")
before = correction_snapshot.table("orbit_correction_before")
after = correction_snapshot.table("orbit_correction_after")

assert {"snapshot_id", "correction_label", "plane", "rms_change_mm"}.issubset(summary.columns)
assert {"delta_current_A", "delta_kick_rad", "delta_kick_mrad", "enabled"}.issubset(correctors.columns)
assert {"before_model_mm", "before_residual_mm", "after_model_mm", "after_residual_mm", "residual_change_mm"}.issubset(bpm_compare.columns)
assert {"rms_orbit_mm", "max_abs_orbit_mm"}.issubset(before.columns)
assert {"rms_orbit_mm", "max_abs_orbit_mm"}.issubset(after.columns)
assert correctors.loc[correctors["enabled"], "delta_current_A"].abs().max() > 0.0
assert bpm_compare["after_model_mm"].notna().any()
assert correction_snapshot.table("warnings")["severity"].isin(["warning", "error"]).all() or correction_snapshot.table("warnings").empty

summary[["correction_label", "plane", "n_enabled_correctors", "rms_change_mm"]]


In [ ]:
h_correction = next(result.result for result in correction_snapshot.orbit_correction_results if result.plane == "H")
v_correction = next(result.result for result in correction_snapshot.orbit_correction_results if result.plane == "V")

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)
plot_orbit_with_bpm(
    h_correction.measured_twiss_df,
    h_correction.bpm_measurements,
    plane="H",
    ax=axes[0],
    label="Measured fitted orbit",
    title="Horizontal measured and corrected orbit suggestion",
)
plot_orbit_with_bpm(
    h_correction.corrected_twiss_df,
    h_correction.bpm_measurements,
    plane="H",
    ax=axes[0],
    label="Corrected orbit suggestion",
    orbit_kwargs={"linestyle": "--"},
)
plot_orbit_with_bpm(
    v_correction.measured_twiss_df,
    v_correction.bpm_measurements,
    plane="V",
    ax=axes[1],
    label="Measured fitted orbit",
    title="Vertical measured and corrected orbit suggestion",
)
plot_orbit_with_bpm(
    v_correction.corrected_twiss_df,
    v_correction.bpm_measurements,
    plane="V",
    ax=axes[1],
    label="Corrected orbit suggestion",
    orbit_kwargs={"linestyle": "--"},
)
fig.tight_layout()
save_snapshot_plot(correction_snapshot, fig, plot_name="corrected_orbit_suggestions", aspect="correction")
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)
plot_corrector_suggestions(
    h_correction.correctors,
    ax=axes[0],
    value="delta_current_A",
    title="Horizontal MAD-X CORRECT current suggestions",
)
plot_corrector_suggestions(
    v_correction.correctors,
    ax=axes[1],
    value="delta_current_A",
    title="Vertical MAD-X CORRECT current suggestions",
)
fig.tight_layout()
save_snapshot_plot(correction_snapshot, fig, plot_name="corrector_current_suggestions", aspect="correction")
plt.show()

manifest = correction_snapshot.table("plot_manifest")
assert len(manifest) == 2
assert manifest["dpi"].eq(200).all()
assert manifest["has_title"].all()
assert manifest["has_axis_labels"].all()
assert manifest["has_legend"].all()
manifest[["aspect", "plot_name", "path", "dpi"]]


In [ ]:
bpm_compare[[
    "correction_label",
    "plane",
    "bpm",
    "measurement_mm",
    "before_model_mm",
    "after_model_mm",
    "before_residual_mm",
    "after_residual_mm",
    "residual_change_mm",
]].head(12)
